In [0]:
print("Pipeline started")

Pipeline started


In [0]:

%sql
CREATE SCHEMA IF NOT EXISTS weather_project;
USE weather_project;

In [0]:
from pyspark.sql import Row

stores = [
    Row(store_id=101, city="Toronto", country="Canada", latitude=43.6532, longitude=-79.3832),
    Row(store_id=102, city="New York", country="USA", latitude=40.7128, longitude=-74.0060),
    Row(store_id=103, city="Chicago", country="USA", latitude=41.8781, longitude=-87.6298),
    Row(store_id=104, city="London", country="UK", latitude=51.5072, longitude=-0.1276),
    Row(store_id=105, city="Mumbai", country="India", latitude=19.0760, longitude=72.8777)
]

df_stores = spark.createDataFrame(stores)

df_stores.write.mode("overwrite").format("delta").saveAsTable("weather_project.bronze_stores")

In [0]:
%sql
SELECT * FROM weather_project.bronze_stores;

store_id,city,country,latitude,longitude
101,Toronto,Canada,43.6532,-79.3832
102,New York,USA,40.7128,-74.006
103,Chicago,USA,41.8781,-87.6298
104,London,UK,51.5072,-0.1276
105,Mumbai,India,19.076,72.8777


In [0]:
import requests
from datetime import datetime, timezone
from pyspark.sql import Row

stores = spark.table("weather_project.bronze_stores").collect()

weather_rows = []

for store in stores:
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={store.latitude}"
        f"&longitude={store.longitude}"
        "&current=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"
    )

    response = requests.get(url)
    data = response.json()

    current = data.get("current", {})

    weather_rows.append(Row(
        store_id=store.store_id,
        city=store.city,
        country=store.country,
        latitude=store.latitude,
        longitude=store.longitude,
        api_capture_time=datetime.now(timezone.utc).isoformat(),
        weather_time=current.get("time"),
        temperature_2m=current.get("temperature_2m"),
        humidity=current.get("relative_humidity_2m"),
        precipitation=current.get("precipitation"),
        wind_speed_10m=current.get("wind_speed_10m")
    ))

df_weather = spark.createDataFrame(weather_rows)

df_weather.write.mode("append").format("delta").saveAsTable("weather_project.bronze_weather_api")

In [0]:
display(df_weather )

store_id,city,country,latitude,longitude,api_capture_time,weather_time,temperature_2m,humidity,precipitation,wind_speed_10m
101,Toronto,Canada,43.6532,-79.3832,2026-04-29T01:19:28.951636+00:00,2026-04-29T01:15,10.3,93,0.0,1.6
102,New York,USA,40.7128,-74.006,2026-04-29T01:19:29.397590+00:00,2026-04-29T01:15,11.3,57,0.0,6.7
103,Chicago,USA,41.8781,-87.6298,2026-04-29T01:19:29.842899+00:00,2026-04-29T01:15,12.5,83,0.0,6.6
104,London,UK,51.5072,-0.1276,2026-04-29T01:19:30.288775+00:00,2026-04-29T01:15,9.2,70,0.0,19.1
105,Mumbai,India,19.076,72.8777,2026-04-29T01:19:30.751816+00:00,2026-04-29T01:15,26.5,95,0.0,5.4


In [0]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp

df_bronze_weather = spark.table("weather_project.bronze_weather_api")

df_silver_weather = (
    df_bronze_weather
    .withColumn("api_capture_ts", to_timestamp("api_capture_time"))
    .withColumn("weather_ts", to_timestamp("weather_time"))
    .withColumn("temperature_2m", col("temperature_2m").cast("double"))
    .withColumn("humidity", col("humidity").cast("double"))
    .withColumn("precipitation", col("precipitation").cast("double"))
    .withColumn("wind_speed_10m", col("wind_speed_10m").cast("double"))
    .withColumn("processed_at", current_timestamp())
    .dropDuplicates(["store_id", "weather_time", "api_capture_time"])
)

df_silver_weather.write.mode("overwrite").format("delta").saveAsTable("weather_project.silver_weather")

In [0]:
%sql
CREATE OR REPLACE TABLE weather_project.gold_store_weather_status AS
SELECT
    store_id,
    city,
    country,
    weather_ts,
    temperature_2m,
    humidity,
    precipitation,
    wind_speed_10m,
    CASE
        WHEN precipitation > 0 THEN 'RAIN_ALERT'
        WHEN temperature_2m >= 30 THEN 'HEAT_ALERT'
        WHEN wind_speed_10m >= 30 THEN 'WIND_ALERT'
        ELSE 'NORMAL'
    END AS weather_status,
    processed_at
FROM weather_project.silver_weather;

num_affected_rows,num_inserted_rows


In [0]:
print("Pipeline finished successfully")

Pipeline finished successfully


In [0]:
%sql
SELECT * 
FROM weather_project.gold_store_weather_status
ORDER BY processed_at DESC;

store_id,city,country,weather_ts,temperature_2m,humidity,precipitation,wind_speed_10m,weather_status,processed_at
101,Toronto,Canada,2026-04-29T01:15:00.000Z,10.3,93.0,0.0,1.6,NORMAL,2026-04-29T01:38:11.202Z
104,London,UK,2026-04-29T01:15:00.000Z,9.2,70.0,0.0,19.1,NORMAL,2026-04-29T01:38:11.202Z
105,Mumbai,India,2026-04-29T01:15:00.000Z,26.5,95.0,0.0,5.4,NORMAL,2026-04-29T01:38:11.202Z
103,Chicago,USA,2026-04-29T01:15:00.000Z,12.5,83.0,0.0,6.6,NORMAL,2026-04-29T01:38:11.202Z
102,New York,USA,2026-04-29T01:15:00.000Z,11.3,57.0,0.0,6.7,NORMAL,2026-04-29T01:38:11.202Z
